# Rolling MPS vs Classical Indicators — Cross-Asset (no Bitcoin, 2007-2026, incl. 2008 GFC)
**MFE Industry Project · AQS**

MPS-derived indicators (fit on rolling 252-day windows, fixed Fiedler order) compared head-to-head with the
standard risk indicators, on the 12-asset diversified book **without Bitcoin** (CRSP equities + small-cap + intl + rates + IG/HY credit + gold + commodities + REITs), reaching back through the 2008 Global Financial Crisis.

Classical set: **realized volatility, average pairwise correlation, first-PC explained variance (absorption
ratio), VIX, MOVE, drawdown, realized VaR breaches, crisis periods**. Questions: do the MPS indicators track
the classical ones (validation), flag crises, and carry **non-redundant** signal beyond vol/correlation/absorption?

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, sys
here = os.getcwd(); ROOT = here if os.path.isdir(os.path.join(here,"src")) else os.path.dirname(here)
def P(rel): return os.path.join(ROOT, rel)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from numpy.linalg import lstsq
CRISES = [('2008-09-01', '2009-03-31', 'GFC-2008'), ('2011-07-01', '2011-10-31', 'euro'), ('2015-08-01', '2016-02-29', 'China/oil'), ('2018-10-01', '2018-12-31', 'Q4-2018'), ('2020-02-15', '2020-04-30', 'COVID'), ('2022-01-01', '2022-10-31', 'rate-hikes'), ('2023-03-01', '2023-03-31', 'SVB')]
df = pd.read_csv(P("data/rolling_gfc.csv"), index_col=0, parse_dates=True)
mps_cols = [c for c in df.columns if c.startswith("mps_")]
cls_cols = ["realized_vol","avg_corr","pc1_share","VIX","MOVE","drawdown","var_breaches"]
print("rolling table:", df.shape, "|", df.index.min().date(), "->", df.index.max().date())
print(df[["mps_entanglement","mps_total_corr","mps_tail_state","realized_vol","avg_corr","pc1_share","VIX"]].describe().round(3).to_string())

rolling table: (219, 14) | 2008-04-11 -> 2026-06-24
       mps_entanglement  mps_total_corr  mps_tail_state  realized_vol  avg_corr  pc1_share      VIX
count           219.000         219.000         219.000       219.000   219.000    219.000  219.000
mean              2.050           7.978           0.000         0.142     0.327      0.499   19.964
std               0.008           0.570           0.000         0.066     0.072      0.068    6.186
min               2.024           7.076           0.000         0.067     0.154      0.340   11.020
25%               2.045           7.492           0.000         0.099     0.278      0.454   15.162
50%               2.050           7.828           0.000         0.119     0.321      0.493   18.574
75%               2.056           8.423           0.000         0.166     0.370      0.554   23.316
max               2.068           9.417           0.001         0.372     0.505      0.654   41.026


## 1 · Overlay — do the MPS indicators move with market stress? (crisis windows shaded)

In [2]:
def z(s): return (s - s.mean())/s.std()
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
panels = [("MPS entanglement vs VIX & absorption (PC1)", ["mps_entanglement","VIX","pc1_share"]),
          ("MPS total-correlation & mean-MI vs avg-corr & MOVE", ["mps_total_corr","mps_mean_MI","avg_corr","MOVE"]),
          ("MPS tail-state prob & tail-asym vs drawdown & VaR breaches", ["mps_tail_state","mps_tail_asym","drawdown","var_breaches"])]
for ax,(title,keys) in zip(axes, panels):
    for k in keys: ax.plot(df.index, z(df[k]), label=k, lw=1.5)
    for s,e,lab in CRISES:
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color="red", alpha=0.10)
        ax.text(pd.Timestamp(s), ax.get_ylim()[1]*0.9, lab, fontsize=7, color="darkred")
    ax.set_title(title); ax.legend(fontsize=8, ncol=4); ax.axhline(0, color="k", lw=0.4)
axes[-1].set_xlabel("window-end date"); plt.tight_layout(); plt.savefig(P("data/rv_overlay.png"), dpi=110); plt.show()

## 2 · Spearman correlation — MPS features vs classical indicators
High |ρ| ⇒ the MPS feature re-expresses a classical indicator (redundant, but validates it). Uniformly low ⇒
it measures something else.

In [3]:
M = pd.DataFrame(index=mps_cols, columns=cls_cols, dtype=float)
for a in mps_cols:
    for b in cls_cols: M.loc[a,b] = spearmanr(df[a], df[b]).correlation
fig, ax = plt.subplots(figsize=(10,5.5)); im = ax.imshow(M.values.astype(float), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cls_cols))); ax.set_xticklabels(cls_cols, rotation=45, ha="right")
ax.set_yticks(range(len(mps_cols))); ax.set_yticklabels(mps_cols)
for i in range(len(mps_cols)):
    for j in range(len(cls_cols)): ax.text(j,i,"%.2f"%M.values[i,j], ha="center", va="center", fontsize=7)
plt.colorbar(im, fraction=0.046); ax.set_title("Spearman: MPS features (rows) vs classical (cols)")
plt.tight_layout(); plt.savefig(P("data/rv_corrtable.png"), dpi=110); plt.show()
print(M.astype(float).round(2).to_string())

                  realized_vol  avg_corr  pc1_share   VIX  MOVE  drawdown  var_breaches
mps_entanglement         -0.06     -0.14      -0.13 -0.05 -0.00      0.04         -0.04
mps_total_corr            0.78      0.33       0.85  0.75  0.46     -0.42          0.23
mps_mean_MI               0.80      0.36       0.89  0.78  0.41     -0.44          0.25
mps_tail_state            0.20      0.35       0.20  0.12 -0.19      0.06          0.50
mps_tail_asym             0.17      0.26       0.17  0.14 -0.11      0.09          0.12
mps_corr_len              0.33      0.19       0.38  0.32  0.18     -0.14          0.11
mps_bond_dim             -0.06     -0.14      -0.13 -0.05 -0.00      0.04         -0.04


## 3 · Lead–lag vs VIX — does any MPS indicator move *before* VIX?

In [4]:
def leadlag(a,b,maxlag=6):
    a=a.diff().dropna(); b=b.diff().dropna(); idx=a.index.intersection(b.index); a,b=a.loc[idx],b.loc[idx]
    return pd.Series({L: a.corr(b.shift(-L)) for L in range(-maxlag,maxlag+1)})
fig, ax = plt.subplots(figsize=(9,4))
for feat in ["mps_entanglement","mps_total_corr","mps_tail_state"]:
    ll = leadlag(df[feat], df["VIX"]); ax.plot(ll.index, ll.values, marker="o", label=feat)
ax.axvline(0, color="k", lw=0.5); ax.set_xlabel("lag (months): <0 = MPS leads VIX"); ax.set_ylabel("corr of monthly Δ")
ax.set_title("Lead-lag vs VIX"); ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(P("data/rv_leadlag.png"), dpi=110); plt.show()
for feat in ["mps_entanglement","mps_total_corr","mps_tail_state","mps_tail_asym"]:
    ll = leadlag(df[feat], df["VIX"]); L=ll.abs().idxmax(); print("%-18s peak |corr| at lag %+d (corr=%.2f)  [neg=leads]" % (feat, L, ll[L]))

mps_entanglement   peak |corr| at lag -3 (corr=0.09)  [neg=leads]
mps_total_corr     peak |corr| at lag +0 (corr=0.41)  [neg=leads]
mps_tail_state     peak |corr| at lag +1 (corr=0.24)  [neg=leads]
mps_tail_asym      peak |corr| at lag +2 (corr=0.09)  [neg=leads]


## 4 · Redundancy — UNIQUE signal beyond vol / correlation / absorption
Regress each MPS feature on {realized_vol, avg_corr, pc1_share}. **1−R² = unique variation** the classical
trio can't explain. High 1−R² ⇒ a non-redundant indicator worth recommending.

In [5]:
def z(s): return (s-s.mean())/s.std()
Xc = np.column_stack([np.ones(len(df)), z(df["realized_vol"]), z(df["avg_corr"]), z(df["pc1_share"])])
red = []
for a in mps_cols:
    y = z(df[a]).values; beta,_,_,_ = lstsq(Xc, y, rcond=None); yhat=Xc@beta
    r2 = 1-((y-yhat)**2).sum()/((y-y.mean())**2).sum(); red.append((a, r2, 1-r2))
redf = pd.DataFrame(red, columns=["mps_feature","R2_by_classical","unique_1minusR2"]).set_index("mps_feature").sort_values("unique_1minusR2", ascending=False)
print(redf.round(3).to_string())
fig, ax = plt.subplots(figsize=(9,4)); ax.barh(redf.index, redf["unique_1minusR2"], color="#2c7fb8")
ax.set_xlabel("unique variation (1 - R^2) not explained by vol/corr/absorption"); ax.set_title("Non-redundancy of MPS indicators")
plt.tight_layout(); plt.savefig(P("data/rv_redundancy.png"), dpi=110); plt.show()

                  R2_by_classical  unique_1minusR2
mps_feature                                       
mps_bond_dim                0.042            0.958
mps_entanglement            0.042            0.958
mps_tail_asym               0.052            0.948
mps_corr_len                0.162            0.838
mps_tail_state              0.205            0.795
mps_total_corr              0.806            0.194
mps_mean_MI                 0.853            0.147


## 5 · Crisis vs calm — which indicators discriminate stress?

In [6]:
flag = pd.Series(False, index=df.index)
for s,e,_ in CRISES: flag |= (df.index>=pd.Timestamp(s)) & (df.index<=pd.Timestamp(e))
comp = pd.DataFrame({"calm_mean": df[~flag].mean(), "crisis_mean": df[flag].mean()})
comp["separation"] = (df[flag].mean()-df[~flag].mean())/df.std()
print("windows: %d crisis, %d calm" % (int(flag.sum()), int((~flag).sum())))
print(comp.loc[mps_cols+cls_cols].round(3).to_string())
fig, ax = plt.subplots(figsize=(9,5)); sep = comp.loc[mps_cols+cls_cols,"separation"].sort_values()
colors = ["#2c7fb8" if i.startswith("mps_") else "#999999" for i in sep.index]
ax.barh(sep.index, sep.values, color=colors); ax.set_xlabel("standardized crisis - calm gap (blue=MPS)")
ax.set_title("Crisis discrimination"); plt.tight_layout(); plt.savefig(P("data/rv_crisis.png"), dpi=110); plt.show()

windows: 34 crisis, 185 calm
                  calm_mean  crisis_mean  separation
mps_entanglement      2.050        2.049      -0.083
mps_total_corr        7.960        8.079       0.210
mps_mean_MI           0.158        0.163       0.133
mps_tail_state        0.000        0.000       0.044
mps_tail_asym         0.049        0.025      -0.363
mps_corr_len          1.050        1.064       0.193
mps_bond_dim          7.769        7.764      -0.082
realized_vol          0.138        0.164       0.398
avg_corr              0.328        0.320      -0.109
pc1_share             0.498        0.505       0.106
VIX                  19.680       21.512       0.296
MOVE                 85.271       94.505       0.328
drawdown             -0.044       -0.161      -1.385
var_breaches          2.746        5.206       1.017


## 6 · Verdict — feature recommendation
*(Read against the tables/plots above.)*
- **Validation:** MPS features with high |ρ| to VIX/absorption/avg-corr (§2) confirm the MPS re-derives standard
  risk structure from the joint density — a sanity check, not a weakness.
- **Non-redundant features (§4):** those with the highest **1−R²** carry signal beyond vol/correlation/absorption
  — the genuine value-add (expect the higher-order/tail features: tail-state prob, tail-asymmetry, correlation
  length, which have no linear-risk analogue).
- **Crisis discrimination (§5):** features with the largest standardized crisis−calm gap are the best stress flags.
- **Lead-lag (§3):** report honestly whether any MPS feature *leads* VIX (negative-lag peak) or is coincident.
- **Recommendation:** keep the MPS indicators that are BOTH crisis-sensitive AND non-redundant; drop any that are
  both redundant (high R²) and crisis-insensitive.